# FastSpeech 2 — English (Svarah) / Telugu (IndicTTS) — GPU runbook

Runs the complete pipeline for **one language per session**:
smoke test → data download & selection → MFA alignment → feature extraction → training → test-sentence synthesis + variance-adaptor plots → verification.

**Fastest route:** open this notebook twice (two Colab sessions/accounts, or Colab + Kaggle), set `LANG = "English"` in one and `LANG = "Telugu"` in the other, and run all cells.

* Runtime → Change runtime type → **T4 GPU**.
* Checkpoints, logs and `Results/<LANG>` are written to Google Drive, so a disconnect loses nothing: re-run all cells and training resumes from `latest.pth.tar`.
* English needs a Hugging Face token (Svarah): https://huggingface.co/settings/tokens

In [ ]:
# ---- parameters -------------------------------------------------------------
LANG = "English"            # "English" or "Telugu"
MAX_TRAIN_HOURS = 3.5       # wall-clock budget for train.py (it saves and stops cleanly); None = full 40k steps
DRIVE_ROOT = "/content/drive/MyDrive/fastspeech2_assignment"
REPO_URL = ""               # your GitHub repo URL; leave "" to unzip DRIVE_ROOT/fastspeech2_project.zip instead
assert LANG in ("English", "Telugu")

In [ ]:
import os, subprocess, shutil, sys
def sh(cmd, check=True):
    """Run a shell command and stream its output (stdout+stderr) into the cell."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError(f"command failed ({rc}): {cmd}  <- the real error is printed above")

sh("nvidia-smi", check=False)
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)

## 1. Get the project and persist outputs on Drive

In [ ]:
P = "/content/project"
if not os.path.exists(P):
    if REPO_URL:
        sh(f"git clone {REPO_URL} {P}")
    else:
        sh(f'unzip -q "{DRIVE_ROOT}/fastspeech2_project.zip" -d /content/')
os.chdir(P)

# checkpoints / logs / Results/<LANG> live on Drive (symlinked), so nothing is lost on disconnect
for sub in [f"{LANG}/checkpoints", f"{LANG}/logs", f"Results/{LANG}"]:
    local, remote = os.path.join(P, sub), os.path.join(DRIVE_ROOT, sub)
    if os.path.islink(local):
        continue
    os.makedirs(remote, exist_ok=True)
    if os.path.isdir(local):
        shutil.copytree(local, remote, dirs_exist_ok=True)
        shutil.rmtree(local)
    os.symlink(remote, local)
for d in ["test_generated_audios", "variance_adaptor_plots/duration", "variance_adaptor_plots/pitch", "variance_adaptor_plots/energy"]:
    os.makedirs(f"Results/{LANG}/{d}", exist_ok=True)
sh("ls -la")

## 2. Python dependencies + HiFi-GAN vocoder

In [ ]:
sh("pip install -q -r requirements.txt")
# Telugu-capable font so plot titles can show the Telugu sentence (phone labels are ASCII either way)
sh("apt-get -qq install -y fonts-noto-core > /dev/null && rm -rf ~/.cache/matplotlib", check=False)
sh("python -c 'import pyworld, g2p_en, datasets, soundfile; print(\"deps OK\")'")
# universal HiFi-GAN released with the reference implementation (ming024/FastSpeech2)
if not os.path.exists("vocoder/generator_universal.pth.tar"):
    sh("git clone --depth 1 https://github.com/ming024/FastSpeech2 /content/ming024_FastSpeech2")
    sh("unzip -o /content/ming024_FastSpeech2/hifigan/generator_universal.pth.tar.zip -d vocoder/")
sh("ls -la vocoder")

## 3. Smoke test (≈2 min): synthetic data through preprocessing → training → resume → inference

In [ ]:
sh(f"python tools/smoke_test.py --language {LANG.lower()} --steps 40")

## 4. Hugging Face login (required for Svarah; harmless for Telugu)

In [ ]:
from huggingface_hub import login
login()   # paste a read token; accept the dataset terms on its HF page first if asked

## 5. Montreal Forced Aligner (conda-forge, installed with micromamba — no runtime restart)

In [ ]:
MAMBA = "/usr/local/bin/micromamba"
if not os.path.exists(MAMBA):
    sh("curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /usr/local bin/micromamba")
if not os.path.exists("/root/micromamba/envs/mfa"):
    sh(f"{MAMBA} create -y -q -r /root/micromamba -n mfa -c conda-forge montreal-forced-aligner")
os.environ["MFA_CMD"] = f"{MAMBA} run -r /root/micromamba -n mfa mfa"
sh(os.environ["MFA_CMD"] + " version")

## 6. Preprocessing
Data, TextGrids and features are kept on the fast local disk; the finished features are archived to Drive so a new session can skip this section (cell 6d).

In [ ]:
# 6a. download + 5 h / 1 h selection + MFA corpus + lexicon
if not os.path.exists(f"{LANG}/data/corpus/split.json"):
    sh(f"python {LANG}/preprocess.py --stage prepare")
sh(f"cat {LANG}/data/corpus/selection_summary.json")

In [ ]:
# 6b. forced alignment (English: pretrained english_us_arpa; Telugu: MFA model trained on the corpus)
sh(f"python {LANG}/preprocess.py --stage align")

In [ ]:
# 6c. features + archive to Drive
sh(f"python {LANG}/preprocess.py --stage features")
sh(f"cat {LANG}/data/preprocessed/preprocess_summary.json")
sh(f'tar -czf "{DRIVE_ROOT}/{LANG}_preprocessed.tgz" -C {LANG}/data preprocessed')

In [ ]:
# 6d. NEW SESSION ONLY: restore features from Drive instead of re-running 6a-6c
arch = f"{DRIVE_ROOT}/{LANG}_preprocessed.tgz"
if not os.path.exists(f"{LANG}/data/preprocessed/train.txt") and os.path.exists(arch):
    os.makedirs(f"{LANG}/data", exist_ok=True)
    sh(f'tar -xzf "{arch}" -C {LANG}/data')
print(os.path.exists(f"{LANG}/data/preprocessed/train.txt"))

## 7. Training (resumable — just re-run this cell after a disconnect)

In [ ]:
extra = f"--max_hours {MAX_TRAIN_HOURS}" if MAX_TRAIN_HOURS else ""
sh(f"python {LANG}/train.py --restore latest {extra}")

In [ ]:
from IPython.display import Image, display
display(Image(f"Results/{LANG}/training_loss.png")); display(Image(f"Results/{LANG}/validation_loss.png"))

## 8. Synthesize every test sentence + variance-adaptor plots

In [ ]:
sh(f"python {LANG}/inference.py --test_set")
import json, glob
from IPython.display import Audio
info = json.load(open(f"Results/{LANG}/predictions.json", encoding="utf-8"))
print("checkpoint:", info["checkpoint"], "step", info["checkpoint_step"], "| vocoder:", info["vocoder"])
for sid, r in info["sentences"].items():
    print(sid, r["text"]); display(Audio(f"Results/{LANG}/test_generated_audios/{sid}.wav"))
    for k in ("duration", "pitch", "energy"):
        display(Image(f"Results/{LANG}/variance_adaptor_plots/{k}/{sid}_{k}.png"))

## 9. Verify and collect

In [ ]:
sh("python tools/verify_deliverables.py", check=False)
sh(f"ls -la {LANG}/checkpoints")
print("Everything under Results/ and the checkpoints is already on Drive at", DRIVE_ROOT)

### Running on Kaggle instead
Same cells, with these changes: skip the Drive cells, set `P = "/kaggle/working/project"` and upload the project zip as a Kaggle dataset; add your HF token as a Kaggle secret and call `login(token=...)`. Enable *GPU T4 x2* and *Internet*. Save `/kaggle/working` as notebook output to keep checkpoints.